# Tutorial 8 — RNA-Seq QC & Differential Expression
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

RNA-Seq measures gene expression genome-wide. In toxicogenomics, we compare treated vs control samples to identify differentially expressed genes (DEGs) that reveal mechanisms of toxicity — the approach used in my BHSAI publications on liver and kidney injury.

In [ ]:
!pip install pydeseq2 pandas numpy matplotlib seaborn scipy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
n_genes, n_ctrl, n_treat = 2000, 3, 3
sample_names = [f"Control_{i+1}" for i in range(n_ctrl)] + [f"Treated_{i+1}" for i in range(n_treat)]
gene_names   = [f"Gene_{i:05d}" for i in range(n_genes)]

# Simulate realistic count data
base_counts = np.random.negative_binomial(8, 0.4, size=(n_genes, n_ctrl+n_treat))
# Introduce DE: 100 up, 60 down
base_counts[50:150, n_ctrl:] = (base_counts[50:150, n_ctrl:] * np.random.uniform(3, 8, (100, n_treat))).astype(int)
base_counts[150:210, n_ctrl:] = (base_counts[150:210, n_ctrl:] * np.random.uniform(0.1, 0.3, (60, n_treat))).astype(int)

counts = pd.DataFrame(base_counts, index=gene_names, columns=sample_names)
print(f"Count matrix: {counts.shape[0]:,} genes × {counts.shape[1]} samples")
print(f"Library sizes:")
for col in counts.columns:
    print(f"  {col}: {counts[col].sum():>12,} reads")

## QC: filter low-count genes

In [ ]:
min_count = 10
keep = counts.sum(axis=1) >= min_count
counts_f = counts[keep]
print(f"Genes before filter: {len(counts):,}")
print(f"Genes after  filter: {len(counts_f):,} ({len(counts_f)/len(counts)*100:.1f}% retained)")

# Library size distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
counts.sum().plot(kind="bar", ax=ax1, color="#1565c0", edgecolor="white")
ax1.set_title("Library sizes per sample"); ax1.set_ylabel("Total counts"); ax1.tick_params(axis='x', rotation=30)

log_mean = np.log2(counts_f.mean(axis=1) + 1)
ax2.hist(log_mean, bins=60, color="#00897b", edgecolor="white")
ax2.set_title("log2(mean count + 1) distribution")
ax2.set_xlabel("log2(mean count + 1)"); ax2.set_ylabel("# genes")
plt.tight_layout(); plt.savefig("rnaseq_qc.png", dpi=150); plt.show()

## DESeq2-style differential expression (simplified)

In [ ]:
def simple_de(counts_df, ctrl_cols, treat_cols):
    rows = []
    for gene in counts_df.index:
        ctrl  = counts_df.loc[gene, ctrl_cols].values.astype(float) + 0.5
        treat = counts_df.loc[gene, treat_cols].values.astype(float) + 0.5
        lfc   = np.log2(treat.mean()) - np.log2(ctrl.mean())
        _, pval = stats.ttest_ind(np.log2(treat), np.log2(ctrl))
        rows.append({"gene": gene, "baseMean": (ctrl.mean()+treat.mean())/2,
                     "log2FoldChange": lfc, "pvalue": pval})
    res = pd.DataFrame(rows).set_index("gene")
    from statsmodels.stats.multitest import multipletests
    res["padj"] = multipletests(res["pvalue"].fillna(1), method="fdr_bh")[1]
    return res.dropna().sort_values("padj")

try:
    from statsmodels.stats.multitest import multipletests
    ctrl_cols  = [c for c in counts_f.columns if "Control" in c]
    treat_cols = [c for c in counts_f.columns if "Treated" in c]
    results = simple_de(counts_f, ctrl_cols, treat_cols)
    sig = results[(results["padj"] < 0.05) & (results["log2FoldChange"].abs() > 1)]
    print(f"Significant DEGs (padj<0.05, |LFC|>1): {len(sig)}")
    print(f"  Upregulated:   {(sig['log2FoldChange']>0).sum()}")
    print(f"  Downregulated: {(sig['log2FoldChange']<0).sum()}")
    print(sig.head(10)[["baseMean","log2FoldChange","pvalue","padj"]].round(4))
except ImportError:
    print("Install statsmodels: pip install statsmodels")
    results = pd.DataFrame()

## Volcano plot

In [ ]:
if len(results) > 0:
    pval_thresh = 0.05; lfc_thresh = 1.0
    up   = (results["log2FoldChange"] >  lfc_thresh) & (results["padj"] < pval_thresh)
    down = (results["log2FoldChange"] < -lfc_thresh) & (results["padj"] < pval_thresh)
    ns   = ~up & ~down

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(results.loc[ns,  "log2FoldChange"], -np.log10(results.loc[ns,  "padj"]+1e-300), c="#b0b0b0", s=10, alpha=0.5, label=f"NS ({ns.sum()})")
    ax.scatter(results.loc[up,  "log2FoldChange"], -np.log10(results.loc[up,  "padj"]+1e-300), c="#c0392b", s=20, alpha=0.85, label=f"Up ({up.sum()})")
    ax.scatter(results.loc[down,"log2FoldChange"], -np.log10(results.loc[down,"padj"]+1e-300), c="#2980b9", s=20, alpha=0.85, label=f"Down ({down.sum()})")
    ax.axhline(-np.log10(pval_thresh), color="k", linestyle="--", lw=0.8)
    ax.axvline(lfc_thresh, color="gray", linestyle="--", lw=0.8)
    ax.axvline(-lfc_thresh, color="gray", linestyle="--", lw=0.8)
    ax.set_xlabel("log2 Fold Change"); ax.set_ylabel("-log10(padj)")
    ax.set_title("Volcano Plot — Treated vs Control"); ax.legend(frameon=False)
    plt.tight_layout(); plt.savefig("volcano.png", dpi=150); plt.show()

## Key takeaways
- Always filter low-count genes before DE analysis
- Use FDR correction (BH method) — never raw p-values for genome-wide testing
- |LFC| > 1 and padj < 0.05 are standard thresholds; adjust based on biology
- See: Goel H et al. *Int. J. Mol. Sci.* 2023, 2024 for real toxicogenomics applications